In [61]:
using LowLevelFEM, LinearAlgebra

In [62]:
openGeometry("tarto.geo")

In [63]:
mat = Material("part", E=2e5, ν=0.3)
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [64]:
A = ∫(U, "hole2", (x,y,z)->1)
Sx = ∫(U, "hole2", (x,y,z)->x)
Sy = ∫(U, "hole2", (x,y,z)->y)
Sz = ∫(U, "hole2", (x,y,z)->z)

xs, ys, zs = Sx/A, Sy/A, Sz/A

(4.005804115633015e-6, -8.595785970749574e-7, 130.0000269311167)

In [65]:
Premote = gmsh.model.occ.addPoint(xs, ys, zs)
gmsh.model.occ.synchronize()
gmsh.model.addPhysicalGroup(0, [Premote], -1, "remote")

gmsh.model.mesh.clear()
gmsh.model.mesh.generate(3)

In [66]:
mat_r = Material("remote")
U = Field([mat, mat_r], type=:VectorField, dim=3, fieldName=:u);
Φ = Field([mat, mat_r], type=:VectorField, dim=3, fieldName=:φ);

In [67]:
Ku = ∫(SymGrad(U)' ⋅ D(:Solid, mat) ⋅ SymGrad(U), Ω="part");

In [68]:
fu = ∫(U ⋅ [0, -5 * A, 0], Γ="remote")
fφ = ∫(Φ ⋅ [0, 0, 0], Γ="remote");

In [69]:
supp1 = BoundaryCondition("base", uz=0, field=U)
supp2 = BoundaryCondition("hole1", ux=0, uy=0, field=U)
supp3 = BoundaryCondition("perim", uz=-0.00001, field=U);

In [70]:
mpc_u = MultiPointConstraint(master="remote", slave="hole2", field=U)
mpc_φ = MultiPointConstraint(master="remote", slave="hole2", field=Φ)
R = rigidRotationMap(mpc_u, mpc_φ);

In [71]:
K = SystemMatrix([Ku Ku * R; R' * Ku R' * Ku * R]);
F = SystemVector([fu, fφ]);

In [72]:
u0, φ = solveField(K, F, support=[supp1, supp2, supp3], mpc=[mpc_u, mpc_φ]);

In [73]:
u = u0 + R * φ;

In [74]:
showDoFResults(fu, name="f")
showDoFResults(u, name="u", visible=true, factor=1);

In [75]:
u = nodesToElements(u, onPhysicalGroup="part");

In [76]:
ε = (u ∘ ∇ + ∇ ∘ u) / 2
showStrainResults(ε, name="ε");

In [77]:
E = mat.E
ν = mat.ν
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν/(1-2ν) * trace(ε) * I)
showDoFResults(σ, name="σ");

In [78]:
λ, Q = eigen(σ);

In [79]:
λ1 = λ[1]
λ2 = λ[2]
λ3 = λ[3];

In [80]:
σ_Mohr = λ1 - λ3
showElementResults(σ_Mohr, name="Mohr");

In [81]:
σ_HMH = √(((λ1 - λ2)^2 + (λ2 - λ3)^2 + (λ3 - λ1)^2) / 2)
showElementResults(σ_HMH, name="von Mises");

In [82]:
showElementResults(σ, :z, name="σz")
showElementResults(σ_Mohr - σ_HMH, name="diff")
showDoFResults((σ_Mohr - σ_HMH) / σ_Mohr * 100, name="rel. diff - smooth")

8

In [83]:
b = -σ⋅∇
showElementResults(b, name="b")

9

In [84]:
openPostProcessor()